In [12]:
from src import MergeData, ExtractFeatures
import pandas as pd
import numpy as np

In [13]:

swat_path = 'data\\SWaT_processed_data.csv'
pcap_path = 'data\\pcap_network_features.csv'

In [19]:
merge = MergeData(swat_path, pcap_path)
df_merged = merge.merge()
print("df_merged shape:", df_merged.shape)

# ograniczenie do wierszy z danymi sieciowymi (okno Ataku 2, ~4356s)
network_cols = ["pkt_count", "total_bytes", "avg_pkt_len"]
df_merged = df_merged.dropna(subset=network_cols).reset_index(drop=True)
print("df_merged po ograniczeniu do okna sieciowego:", df_merged.shape)


df_merged shape: (13201, 88)
df_merged po ograniczeniu do okna sieciowego: (4361, 88)


In [23]:
df_csv = pd.read_csv('data/pcap_network_features.csv')
print("Dane z pcap_network_features.csv:", df_csv.head(5))

Dane z pcap_network_features.csv:    sec_timestamp  pkt_count  total_bytes  avg_pkt_len             datetime
0     1575605700      23504      2646382   112.592835  2019-12-06 04:15:00
1     1575605701      24155      2793602   115.653157  2019-12-06 04:15:01
2     1575605702      24138      2739949   113.511849  2019-12-06 04:15:02
3     1575605703      25476      2765378   108.548359  2019-12-06 04:15:03
4     1575605704      23658      2684782   113.483050  2019-12-06 04:15:04


In [ ]:
ef = ExtractFeatures(
    windows=[5, 10, 30],
    network_cols=["pkt_count", "total_bytes", "avg_pkt_len"],
    process_cols=["LIT101.Pv", "FIT101.Pv"],       # dobierz kluczowe czujniki z Historian
    binary_state_cols=["P101.Status", "MV101.Status"],  # zawory/pompy z Twojego zbioru
    fft_cols=["FIT101.Pv"],                          # nawiązanie do STFT z rozdz. 2
    lag_steps=[1, 5, 10],
    
)
df_out = ef.extract_features(df_merged)

#print(df_out.head())
#print(df_features.describe())
#print(df_out.select_dtypes(include='number').describe().T) 


TypeError: ExtractFeatures.__init__() got an unexpected keyword argument 'fft_window'

In [17]:
df_out.to_csv('data/features_extracted.csv', index=False)

NameError: name 'df_out' is not defined

In [68]:
inf_mask = np.isinf(df_out.select_dtypes(include=[np.number]))
cols_with_inf = inf_mask.sum()
cols_with_inf = cols_with_inf[cols_with_inf > 0].sort_values(ascending=False)
print(cols_with_inf)

corr_avg_pkt_len_FIT101.Pv_10s    2137
corr_avg_pkt_len_FIT101.Pv_5s     1831
corr_avg_pkt_len_FIT101.Pv_30s    1761
corr_total_bytes_FIT101.Pv_30s    1760
corr_pkt_count_FIT101.Pv_10s      1569
corr_pkt_count_FIT101.Pv_30s      1450
corr_total_bytes_FIT101.Pv_5s     1443
corr_pkt_count_FIT101.Pv_5s       1382
corr_total_bytes_FIT101.Pv_10s    1373
corr_pkt_count_LIT101.Pv_5s          1
corr_pkt_count_LIT101.Pv_10s         1
corr_pkt_count_LIT101.Pv_30s         1
dtype: int64


### problem NaN i inf 
wynik standaryzacji Z-score tylko na dobrych wartościach, efekt uboczny metodologii z rozdz 2. 

In [63]:
np.random.seed(0)
n = 300

# symulacja kolumny procesowej ktora ma sigma=0 w kalibracji i inf w momencie ataku
process_with_inf = np.zeros(n)
process_with_inf[250:260] = np.inf   # symulacja realnego zjawiska z ich danych
process_with_inf[260:270] = -np.inf

df = pd.DataFrame({
    'pkt_count': np.random.poisson(24000, n),
    'total_bytes': np.random.normal(2_600_000, 100000, n),
    'avg_pkt_len': np.random.normal(113, 5, n),
    'LIT101.Pv': np.sin(np.linspace(0, 20, n)) + np.random.normal(0, 0.05, n),
    'FIT101.Pv': process_with_inf,
    'P101.Status': np.random.choice([0,1], size=n),
})

ef = ExtractFeatures(
    windows=[5, 10, 30],
    network_cols=['pkt_count', 'total_bytes', 'avg_pkt_len'],
    process_cols=['LIT101.Pv', 'FIT101.Pv'],
    binary_state_cols=['P101.Status'],
    fft_cols=['FIT101.Pv'],
    fft_window=30,
    lag_steps=[1,5,10],
)

out = ef.extract_features(df)
print('---')
print('Shape:', out.shape)
print('NaN total:', out.isna().sum().sum())
print('Inf total:', np.isinf(out.select_dtypes(include=[np.number])).sum().sum())


---
Shape: (300, 163)
NaN total: 206
Inf total: 165


c:\Users\marta\OneDrive\Academia\magisterka_badanie\src\extract_features.py:76: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  for w in self.windows:
c:\Users\marta\OneDrive\Academia\magisterka_badanie\src\extract_features.py:75: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  
c:\Users\marta\OneDrive\Academia\magisterka_badanie\src\extract_features.py:87: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at

In [65]:
assert df_out.isna().sum().sum() == 0, "Znaleziono NaN po ekstrakcji cech!"
assert np.isinf(df_out.select_dtypes(include=[np.number])).sum().sum() == 0, "Znaleziono inf po ekstrakcji cech!"
print("Walidacja jakości danych: OK")

AssertionError: Znaleziono inf po ekstrakcji cech!

In [66]:

print(df_out.head())

   P1_STATE  LIT101.Pv  FIT101.Pv  MV101.Status  P101.Status  P102.Status  \
0 -1.133272   0.836151    -0.3966      -0.25818    -1.291073          0.0   
1 -1.133272   0.839541    -0.3966      -0.25818    -1.291073          0.0   
2 -1.133272   0.833731    -0.3966      -0.25818    -1.291073          0.0   
3 -1.133272   0.833246    -0.3966      -0.25818    -1.291073          0.0   
4 -1.133272   0.835667    -0.3966      -0.25818    -1.291073          0.0   

   P2_STATE  FIT201.Pv  AIT201.Pv  AIT202.Pv  ...  FIT101.Pv_lag_10  \
0       0.0  -1.294814   0.820767   0.463565  ...           -0.3966   
1       0.0  -1.294814   0.820767   0.469404  ...           -0.3966   
2       0.0  -1.294814   0.820767   0.475246  ...           -0.3966   
3       0.0  -1.294814   0.820767   0.478165  ...           -0.3966   
4       0.0  -1.294814   0.820767   0.498607  ...           -0.3966   

   P101.Status_switch_count_5s  P101.Status_switch_count_10s  \
0                          0.0                

In [27]:
print(df_out.describe())

           P1_STATE    LIT101.Pv    FIT101.Pv  MV101.Status  P101.Status  \
count  4.361000e+03  4361.000000  4361.000000   4361.000000  4361.000000   
mean  -1.133272e+00     0.252643     0.181280     -0.066197    -0.171819   
min   -1.133272e+00    -3.145919    -0.396600     -3.962787    -1.291073   
25%   -1.133272e+00     0.028989    -0.396600     -0.258180    -1.291073   
50%   -1.133272e+00     0.625524    -0.396600     -0.258180     0.774550   
75%   -1.133272e+00     0.838089     1.400235     -0.258180     0.774550   
max   -1.133272e+00     1.071957     3.056567      3.446427     0.774550   
std    2.220701e-16     0.964677     1.044127      0.884050     1.029305   

       P102.Status  P2_STATE    FIT201.Pv    AIT201.Pv    AIT202.Pv  ...  \
count       4361.0    4361.0  4361.000000  4361.000000  4361.000000  ...   
mean           0.0       0.0    -0.174627     0.084138    -0.042523  ...   
min            0.0       0.0    -1.295042    -1.178149    -1.365504  ...   
25%        

c:\Users\marta\OneDrive\Academia\magisterka_badanie\.venv\Lib\site-packages\pandas\core\nanops.py:1028: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)
c:\Users\marta\OneDrive\Academia\magisterka_badanie\.venv\Lib\site-packages\numpy\_core\_methods.py:49: RuntimeWarning: invalid value encountered in reduce
  return umr_sum(a, axis, dtype, out, keepdims, initial, where)
c:\Users\marta\OneDrive\Academia\magisterka_badanie\.venv\Lib\site-packages\numpy\_core\_methods.py:49: RuntimeWarning: invalid value encountered in reduce
  return umr_sum(a, axis, dtype, out, keepdims, initial, where)
c:\Users\marta\OneDrive\Academia\magisterka_badanie\.venv\Lib\site-packages\numpy\_core\_methods.py:49: RuntimeWarning: invalid value encountered in reduce
  return umr_sum(a, axis, dtype, out, keepdims, initial, where)
c:\Users\marta\OneDrive\Academia\magisterka_badanie\.venv\Lib\site-packages\pandas\core\nanops.py:1028: RuntimeWarning: invalid value enc